In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path + "/Q1_data.csv")


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
plt.hist(df["Delivery_Time"], bins=20)
plt.xlabel("Delivery Time")
plt.ylabel("Frequency")
plt.title("Delivery Time Distribution")
plt.show()

In [ ]:
df = df.drop(columns=["Order_ID"])


In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

In [ ]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df = pd.get_dummies(df, drop_first=True)


In [ ]:
X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
y = df["Delivery_Time"]

if y.nunique() <= 10:
    counts = y.value_counts()
    ratio = counts.min() / counts.max()

    print("Classification target")
    print("Class counts:\n", counts)
    print("Imbalance ratio:", round(ratio, 2))
else:
    print("Regression target")
    print(y.describe())

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]


In [ ]:
# Task 2,3,4,5: CV split + Train RandomForest + Evaluate using MAE + Print average MAE
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


if y.nunique() <= 10:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits = cv.split(X, y)
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = cv.split(X)

mae_scores = []

for train_idx, test_idx in splits:
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

print("MAE scores per fold:", np.round(mae_scores, 3))
print("Average MAE:", round(np.mean(mae_scores), 3))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Train a final model on the entire dataset X and y
# This model variable will then be used for feature importance
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

feature_importance = pd.Series(
    model.feature_importances_, index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(8,4))
feature_importance.plot(kind="bar")
plt.ylabel("Importance")
plt.title("Feature Importance")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Ensure model is defined if this cell is run independently
# Check if 'model' exists in global scope and is an instance of RandomForestRegressor
if 'model' not in globals() or not isinstance(globals().get('model'), RandomForestRegressor):
    # If not, train a new model for this cell's execution
    model = RandomForestRegressor(random_state=42)
    model.fit(X, y)

y_pred = model.predict(X)

plt.figure(figsize=(6,4))
plt.hist(y_pred, bins=20)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")
plt.show()

In [ ]:
!pip install catboost
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor # Added CatBoostRegressor import

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf = RandomForestRegressor(random_state=42)
    cb = CatBoostRegressor(verbose=0, random_state=42)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    pred_rf = rf.predict(X_test)
    pred_cb = cb.predict(X_test)

    pred_avg = (pred_rf + pred_cb) / 2

    mae = mean_absolute_error(y_test, pred_avg)
    mae_scores.append(mae)
    # Evaluate with MAE on averaged predictions; store per‑fold score

print("MAE per fold:", np.round(mae_scores, 3))
print("Average MAE:", round(np.mean(mae_scores), 3))

# Report fold‑wise MAE (rounded) and the overall cross‑validated average MAE